# Fixed oriented window export (Guiana)

Export training/test `.npy` files from one **fixed rotated 128×128 window** on the full grid
(not random crops). Uses `RotatedFixedCrop` from `src/data/useful_transforms.py`.

Outputs per run:
- `{OUTPUT_PREFIX}_sims.npy` — `(N, 60, 3, 128, 128)` u/v/ssh
- `{OUTPUT_PREFIX}_mask.npy` — `(N, 128, 128)` invalid mask (last conditioning frame)
- `{OUTPUT_PREFIX}_day.npy`, `_hour.npy`, `_latlon.npy`
- `{OUTPUT_PREFIX}_window.npy` — `[top, left, h, w, angle_deg]`

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from joblib import Parallel, delayed
from matplotlib.patches import Polygon

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.dataloaders.guiana_margem_tensor import GuianaDataloader
from src.data.useful_transforms import RotatedFixedCrop, rotated_window_corners

In [ ]:
ZARR_PATH = os.getenv(
    "GUIANA_ZARR_PATH",
    "s3://oceangen-data/margem_equatorial_tensor.zarr",
)

DT = 24
CONDITIONING_STEPS = 4
FORECAST_STEPS = 56
CROP_SIZE = 128

# Fixed oriented window (same for all exports)
WINDOW_TOP = 330
WINDOW_LEFT = 70
WINDOW_ANGLE = -45.0

SEED = 42
N_JOBS = 30
MAX_MASK_FRAC = 0.4  # keep samples with mask.mean() < this

# --- choose period + output prefix ---
# Train export (2016–2019):
# START_TIME = "2016-01-01"
# END_TIME = "2019-12-31"
# OUTPUT_PREFIX = "data/sample_16_19_fixwin"
# N_SAMPLES_TO_GEN = 5000

# Test export (2020–2021):
START_TIME = "2020-01-01"
END_TIME = "2021-12-31"
OUTPUT_PREFIX = "data/sample_20_21_fixwin"
N_SAMPLES_TO_GEN = 1000

Path(OUTPUT_PREFIX).parent.mkdir(parents=True, exist_ok=True)

In [ ]:
fixed_crop = RotatedFixedCrop(
    size=CROP_SIZE,
    top=WINDOW_TOP,
    left=WINDOW_LEFT,
    angle_deg=WINDOW_ANGLE,
)

base_kwargs = dict(
    path=ZARR_PATH,
    dt=DT,
    conditioning_steps=CONDITIONING_STEPS,
    forecast_steps=FORECAST_STEPS,
    start_time=START_TIME,
    end_time=END_TIME,
)

print("Opening Zarr:", ZARR_PATH)
dataset_full = GuianaDataloader(**base_kwargs)
dataset = GuianaDataloader(**base_kwargs, transform=fixed_crop)

info = dataset_full.info()
print("Valid samples:", len(dataset))
print("Grid (H, W):", info["space"]["height"], info["space"]["width"])
print("Window:", fixed_crop)
print("Output prefix:", OUTPUT_PREFIX)

In [ ]:
def stack_time_channels(flat: torch.Tensor, n_frames: int, n_channels: int = 3) -> np.ndarray:
    t = flat.detach().cpu().numpy()
    return t.reshape(n_frames, n_channels, t.shape[-2], t.shape[-1])


def load_sequence(dataset, idx: int):
    x, y, mask = dataset[idx]
    cond = stack_time_channels(y["y"], dataset.conditioning_steps)
    fcst = stack_time_channels(x, dataset.forecast_steps)
    seq = np.concatenate([cond, fcst], axis=0)
    times = dataset.sample_times(idx)
    return seq, mask.detach().cpu().numpy(), times, y


def load_full_sequence(dataset, idx: int):
    return load_sequence(dataset, idx)

## Preview fixed window on full domain

In [ ]:
PREVIEW_IDX = 0

seq_full, mask_full, times, _ = load_full_sequence(dataset_full, PREVIEW_IDX)
seq_crop, mask_crop, _, _ = load_sequence(dataset, PREVIEW_IDX)

H_FULL, W_FULL = seq_full.shape[-2], seq_full.shape[-1]
corners = rotated_window_corners(WINDOW_TOP, WINDOW_LEFT, CROP_SIZE, WINDOW_ANGLE)
t0 = str(times[0])[:16]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
field = seq_full[0, 0]
vmin, vmax = np.nanpercentile(field, 2), np.nanpercentile(field, 98)

ax = axes[0]
ax.imshow(field, cmap="RdBu_r", vmin=vmin, vmax=vmax)
ax.add_patch(Polygon(corners, fill=False, edgecolor="lime", linewidth=2))
ax.set_title(f"Full u @ {t0} | idx={PREVIEW_IDX}")
ax.set_xticks([])
ax.set_yticks([])

ax = axes[1]
ax.imshow(seq_crop[0, 0], cmap="RdBu_r", vmin=vmin, vmax=vmax)
ax.set_title(f"Fixed crop {CROP_SIZE}x{CROP_SIZE} @ {WINDOW_ANGLE}°")
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.show()

print(f"mask invalid fraction (crop): {float(mask_crop.mean()):.4f}")

## Parallel export

In [ ]:
def process_sample(idx):
    """Process one dataset index with the fixed crop."""
    seq, mask, times, y = load_sequence(dataset, int(idx))
    if float(mask.mean()) >= MAX_MASK_FRAC:
        return None

    t_labels = [str(t)[:16] for t in times]
    dt = datetime.strptime(t_labels[3], "%Y-%m-%d %H:%M")
    day_of_year = dt.timetuple().tm_yday
    hour = dt.hour
    latlon = y["latlon"].tolist()

    return seq, mask[0, :, :], day_of_year, hour, latlon


rng = np.random.default_rng(SEED)
n_to_gen = min(N_SAMPLES_TO_GEN, len(dataset))
indices = rng.choice(len(dataset), size=n_to_gen, replace=False)
print(f"Exporting {n_to_gen} samples from {len(dataset)} valid indices...")

results = Parallel(n_jobs=N_JOBS, verbose=10)(
    delayed(process_sample)(idx) for idx in indices
)

results = [r for r in results if r is not None]
if len(results) < n_to_gen:
    print(f"Warning: only {len(results)}/{n_to_gen} passed mask filter (< {MAX_MASK_FRAC} invalid)")

n_out = len(results)
data_sims = np.zeros((n_out, dataset.total_steps, 3, CROP_SIZE, CROP_SIZE), dtype=np.float32)
data_mask = np.zeros((n_out, CROP_SIZE, CROP_SIZE), dtype=np.float32)
data_day = np.zeros((n_out,), dtype=np.int16)
data_hour = np.zeros((n_out,), dtype=np.int8)
data_latlon = np.zeros((n_out, 2), dtype=np.float32)

for i, (seq, mask, day, hour, latlon) in enumerate(results):
    data_sims[i] = seq
    data_mask[i] = mask
    data_day[i] = day
    data_hour[i] = hour
    data_latlon[i] = latlon

window_meta = np.array(
    [WINDOW_TOP, WINDOW_LEFT, CROP_SIZE, CROP_SIZE, WINDOW_ANGLE],
    dtype=np.float64,
)

np.save(f"{OUTPUT_PREFIX}_sims.npy", data_sims)
np.save(f"{OUTPUT_PREFIX}_mask.npy", data_mask)
np.save(f"{OUTPUT_PREFIX}_day.npy", data_day)
np.save(f"{OUTPUT_PREFIX}_hour.npy", data_hour)
np.save(f"{OUTPUT_PREFIX}_latlon.npy", data_latlon)
np.save(f"{OUTPUT_PREFIX}_window.npy", window_meta)

print("Saved:")
print(f"  {OUTPUT_PREFIX}_sims.npy   {data_sims.shape}")
print(f"  {OUTPUT_PREFIX}_mask.npy   {data_mask.shape}")
print(f"  {OUTPUT_PREFIX}_window.npy {window_meta}")